Importation des bibliothèques

In [ ]:
import os
import random
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import copy #pour copier base_model
import numpy as np
import pandas as pd 
import itertools
import csv

In [3]:
from models import SimpleCNN, CNN_MCdropout
from dataset import load_cifar10
from train import train_model, evaluate
from utils import mc_predict_mean_probs, generate_mc_outputs
from accuracy import accuracy_threshold, monotonic_rearrangement, isotonic_regression, monotonicity_penalty

In [4]:
print(os.getcwd())  # donne le répertoire courant

d:\INRIA\MCDropout


Fixation du seed pour la reproductibilité

In [5]:
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [6]:
trainloader, valloader, testloader, classes = load_cifar10(batch_size=128, val_ratio=0.1)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

Chargement du modèle sauvegardé


Test manuel de plusieurs dico_layers et stockage des résultats

In [7]:
layer_names = ["conv1", "conv2", "conv3", "fc1"]
dropout_values = [round(x, 1) for x in np.arange(0.1, 1.01, 0.1)]

grid_configs = []
for r in range(1, len(layer_names)+1):
    for layers in itertools.combinations(layer_names, r):
        for p in dropout_values:
            dico = {layer: p for layer in layers}
            for before in [True, False]:
                grid_configs.append({
                    "dico_layers": dico,
                    "before": before
                })

print(f"Nombre total de configurations : {len(grid_configs)}")
# Exemple d'une config :
print(grid_configs[0])

Nombre total de configurations : 300
{'dico_layers': {'conv1': 0.1}, 'before': True}


Grid search sur dico_layers et stockage des résultats

In [ ]:
all_results = [] 
all_accuracy_curves = []

metrics_list = [
    "variance_predicted",
    "variance_max",
    "predictive_entropy_predicted",
    "predictive_entropy_max",
    "relative_norm"
]

min_penalties = {metric: {"penalty_isotone": float("inf"), "penalty_rearrangement": float("inf"), "config_ids_iso": [], "config_ids_rearr": []} for metric in metrics_list}

X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)

for i, config in enumerate(grid_configs):
    dico_layers = config["dico_layers"]
    before = config["before"]

    model_mc = CNN_MCdropout(copy.deepcopy(base_model), dico_layers=dico_layers, before=before).to(device)
    probs, _ = mc_predict_mean_probs(model_mc, X, T=1000, verbose=False)
    Y_hat = probs.argmax(1)
    acc = (Y_hat == Y).float().mean().item()

    outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(
        model_mc, X, T=1000, metrics=metrics_list, labels=Y, verbose=False
    )

    # Courbes d'accuracy et calcul des pénalités
    penalties = {}
    accuracy_curves = {}
    for metric, color in zip([
        "variance_predicted", "variance_max",
        "predictive_entropy_predicted", "predictive_entropy_max"
    ], ["royalblue", "seagreen", "deeppink", "darkorchid"]):
        thresholds, accuracies = accuracy_threshold(
            Y_hat, Y, metric_values[metric], metric_name=metric, color=color, display=False
        )
        accuracy_curves[metric] = {
            "thresholds": thresholds,
            "accuracies": accuracies
        }
        # Utilisation de monotonicity_penalty à la place
        penalty_iso = monotonicity_penalty(thresholds, accuracies, method="isotonic")
        penalty_rearr = monotonicity_penalty(thresholds, accuracies, method="rearrangement")
        penalties[metric] = {
            "penalty_isotone": penalty_iso,
            "penalty_rearrangement": penalty_rearr
        }

        # Mise à jour des pénalités minimales
        if penalty_iso < min_penalties[metric]["penalty_isotone"]:
            min_penalties[metric]["penalty_isotone"] = penalty_iso
            min_penalties[metric]["config_ids_iso"] = [i]
        elif penalty_iso == min_penalties[metric]["penalty_isotone"]:
            min_penalties[metric]["config_ids_iso"].append(i)

        if penalty_rearr < min_penalties[metric]["penalty_rearrangement"]:
            min_penalties[metric]["penalty_rearrangement"] = penalty_rearr
            min_penalties[metric]["config_ids_rearr"] = [i]
        elif penalty_rearr == min_penalties[metric]["penalty_rearrangement"]:
            min_penalties[metric]["config_ids_rearr"].append(i)

    result = {
        "config_id": i,
        "dico_layers": dico_layers,
        "before": before,
        "acc": acc,
        "mc_mean_probs": mean_probs.mean().item(), 
        **{metric: metric_values[metric].mean().item() if isinstance(metric_values[metric], torch.Tensor) else metric_values[metric] for metric in metrics_list},
        "penalties": penalties
    }
    all_results.append(result)
    all_accuracy_curves.append({
        "config_id": i,
        "accuracy_curves": accuracy_curves
    })

    if (i+1) % 10 == 0 or i == len(grid_configs)-1:
        print(f"Config {i+1}/{len(grid_configs)} traitée.")

# Sauvegarde CSV
df_results = pd.DataFrame(all_results)
df_results.to_csv("all_results.csv", index=False)
print("Résultats sauvegardés dans all_results.csv")

# Affichage des configurations minimisant chaque pénalité
for metric in metrics_list:
    print(f"Métrique '{metric}' :")
    print(f"  - Pénalité isotone minimisée pour config(s) {min_penalties[metric]['config_ids_iso']} avec valeur {min_penalties[metric]['penalty_isotone']}")
    print(f"  - Pénalité rearrangement minimisée pour config(s) {min_penalties[metric]['config_ids_rearr']} avec valeur {min_penalties[metric]['penalty_rearrangement']}")


In [ ]:
import ast

df_results = pd.read_csv("all_results.csv")

# Liste des métriques de pénalité à vérifier
penalty_types = [
    ("variance_predicted", "penalty_isotone"),
    ("variance_predicted", "penalty_rearrangement"),
    ("variance_max", "penalty_isotone"),
    ("variance_max", "penalty_rearrangement"),
    ("predictive_entropy_predicted", "penalty_isotone"),
    ("predictive_entropy_predicted", "penalty_rearrangement"),
    ("predictive_entropy_max", "penalty_isotone"),
    ("predictive_entropy_max", "penalty_rearrangement"),
    # ("relative_norm", "penalty_isotone"),
    # ("relative_norm", "penalty_rearrangement"),
]

for metric, penalty in penalty_types:
    # Extraire la colonne des pénalités (stockée comme dict en string)
    penalties = df_results["penalties"].apply(ast.literal_eval)
    penalty_values = penalties.apply(lambda d: d[metric][penalty])
    min_value = penalty_values.min()
    idx_min = penalty_values[penalty_values == min_value].index

    print(f"\nMétrique '{metric}' avec '{penalty}' minimisée à {min_value:.6f} pour {len(idx_min)} configuration(s) :")
    for idx in idx_min:
        config = df_results.loc[idx]
        print(f"  - config_id: {config['config_id']}, dico_layers: {config['dico_layers']}, before: {config['before']}")